<a href="https://colab.research.google.com/github/KA18202005/DeepLearning-Learning/blob/main/Keras_TUNER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('/content/diabetes.csv')

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [5]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [6]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)

In [7]:
X

array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.20401277,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.68442195,
        -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -1.10325546,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.73518964,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.24020459,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.20212881,
        -0.47378505, -0.87137393]])

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [63]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [11]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.fit(X_train, y_train, batch_size=32, epochs=100, validation_data=(X_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8251 - loss: 0.3885 - val_accuracy: 0.8052 - val_loss: 0.4533
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7942 - loss: 0.4234 - val_accuracy: 0.8052 - val_loss: 0.4535
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8083 - loss: 0.4284 - val_accuracy: 0.8052 - val_loss: 0.4532
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8065 - loss: 0.4284 - val_accuracy: 0.8052 - val_loss: 0.4520
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8151 - loss: 0.4034 - val_accuracy: 0.8052 - val_loss: 0.4540
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8194 - loss: 0.3858 - val_accuracy: 0.8052 - val_loss: 0.4536
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8056 - loss: 0.4052 - val_accuracy: 0.8052 - val_loss: 0.4545
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8118 - loss: 0.4052 - val_accuracy: 0.8117 - 

# **KERAS TUNER**

In [15]:
!pip install keras-tuner
import keras_tuner as kt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.1 MB/s eta 0:00:00


In [24]:
def build_model(hp):

  model = Sequential()
  model.add(Dense(32, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))

  optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta'])

  model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [25]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [26]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [27]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'sgd'}

In [28]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [29]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7275 - loss: 0.5857 - val_accuracy: 0.7987 - val_loss: 0.5623
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7685 - loss: 0.5579 - val_accuracy: 0.7987 - val_loss: 0.5531
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7598 - loss: 0.5477 - val_accuracy: 0.7922 - val_loss: 0.5458
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7202 - loss: 0.5757 - val_accuracy: 0.7987 - val_loss: 0.5382
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7708 - loss: 0.5280 - val_accuracy: 0.7922 - val_loss: 0.5316
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7299 - loss: 0.5442 - val_accuracy: 0.7922 - val_loss: 0.5256
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7375 - loss: 0.5289 - val_accuracy: 0.7922 - val_loss: 0.5201
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7624 - loss: 0.5162 - val_accuracy: 0.79

In [40]:
def build_model(hp):
  model = Sequential()
  units = hp.Int('units', min_value=8, max_value=128, step=8)
  model.add(Dense(units=units, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer = 'sgd', loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [41]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5, directory='mydir', project_name='Kavya')

Reloading Tuner from mydir/Kavya/tuner0.json


In [42]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [43]:
tuner.get_best_hyperparameters()[0].values

{'units': 80}

In [44]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [45]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.6697 - loss: 0.6619 - val_accuracy: 0.7532 - val_loss: 0.6270
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6847 - loss: 0.6422 - val_accuracy: 0.7468 - val_loss: 0.6105
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6978 - loss: 0.6311 - val_accuracy: 0.7597 - val_loss: 0.5965
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6835 - loss: 0.6221 - val_accuracy: 0.7468 - val_loss: 0.5837
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6949 - loss: 0.6025 - val_accuracy: 0.7468 - val_loss: 0.5733
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7328 - loss: 0.5871 - val_accuracy: 0.7532 - val_loss: 0.5633
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7155 - loss: 0.5746 - val_accuracy: 0.7468 - val_loss: 0.5544
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7222 - loss: 0.5791 - val_accuracy: 0.76

In [47]:
def build_model(hp):

  model = Sequential()

  model.add(Dense(80, activation='relu', input_dim=8))

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
    model.add(Dense(80, activation='relu'))

  model.add(Dense(1, activation='sigmoid'))
  model.compile(optimizer = 'sgd', loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [48]:
tuner = kt.RandomSearch(build_model, objective= 'val_accuracy', max_trials=3, directory='mydir', project_name='Kavya1')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [49]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.649350643157959
Total elapsed time: 00h 00m 09s


In [50]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 2}

In [51]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [56]:
model.fit(X_train, y_train, epochs=100, initial_epoch=6, batch_size=32, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7148 - loss: 0.6584 - val_accuracy: 0.6558 - val_loss: 0.6611
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6721 - loss: 0.6519 - val_accuracy: 0.6558 - val_loss: 0.6511
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6494 - loss: 0.6449 - val_accuracy: 0.6494 - val_loss: 0.6417
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6631 - loss: 0.6341 - val_accuracy: 0.6494 - val_loss: 0.6328
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6714 - loss: 0.6208 - val_accuracy: 0.6623 - val_loss: 0.6240
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6753 - loss: 0.6123 - val_accuracy: 0.6753 - val_loss: 0.6147
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7106 - loss: 0.5999 - val_accuracy: 0.6753 - val_loss: 0.6064
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6961 - loss: 0.5930 - val_accuracy: 0.68

In [64]:
def build_model(hp):
  model = Sequential()

  counter = 0

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
    if counter == 0:
      model.add(Dense(hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                      activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid']),
                      input_dim=8))
      model.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])))
    else:
      model.add(Dense(hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                      activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid'])))
      model.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])))

    counter +=1

  model.add(Dense(1, activation='sigmoid'))
  model.compile(optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta', 'nadam']),
                loss='binary_crossentropy', metrics=['accuracy'])
  return model

In [65]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=3, directory='mydir', project_name='final1')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [66]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.6623376607894897

Best val_accuracy So Far: 0.6623376607894897
Total elapsed time: 00h 00m 15s


In [67]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 1,
 'units0': 112,
 'activation0': 'sigmoid',
 'dropout0': 0.2,
 'optimizer': 'sgd',
 'units1': 8,
 'activation1': 'relu',
 'dropout1': 0.3,
 'units2': 72,
 'activation2': 'sigmoid',
 'dropout2': 0.5,
 'units3': 80,
 'activation3': 'sigmoid',
 'dropout3': 0.4,
 'units4': 128,
 'activation4': 'relu',
 'dropout4': 0.3,
 'units5': 104,
 'activation5': 'tanh',
 'dropout5': 0.2,
 'units6': 88,
 'activation6': 'sigmoid',
 'dropout6': 0.4,
 'units7': 16,
 'activation7': 'sigmoid',
 'dropout7': 0.2,
 'units8': 80,
 'activation8': 'sigmoid',
 'dropout8': 0.6}

In [68]:
model = tuner.get_best_models(num_models=1)[0]

In [69]:
model.fit(X_train, y_train, epochs=200, initial_epoch=5, validation_data=(X_test, y_test))

Epoch 6/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6476 - loss: 0.6552 - val_accuracy: 0.6494 - val_loss: 0.6201
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6594 - loss: 0.6279 - val_accuracy: 0.6623 - val_loss: 0.6147
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6709 - loss: 0.6187 - val_accuracy: 0.6688 - val_loss: 0.6110
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6777 - loss: 0.6152 - val_accuracy: 0.6688 - val_loss: 0.6078
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.6746 - loss: 0.6193 - val_accuracy: 0.6623 - val_loss: 0.6059
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6511 - loss: 0.6148 - val_accuracy: 0.6623 - val_loss: 0.6031
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6474 - loss: 0.6228 - val_accuracy: 0.6558 - val_loss: 0.6019
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6694 - loss: 0.6026 - val_accuracy: 0.